# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR² Mass Spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is accessed via its Croissant schema URL (JSON-LD schema):

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

This dataset contains protein abundance, modifications, and sequences data from human mast cell extracellular vesicles obtained through mass spectrometry.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect the description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# View top-level metadata
meta = dataset.metadata
print(f"Dataset name: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}")

## 2. Data Overview
Let's list available record sets, their `@id`s, and their fields and columns, all referenced by `@id`.

_Note: All entity references use `@id` as per the Croissant specification. This enables programmatic reproducibility._

In [ ]:
record_sets = list(meta.record_sets())
print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields (by '@id'):")
    for f in fields:
        print(f"    - {f if isinstance(f, str) else f.get('@id', f)}")
    columns = rs.get('column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        print(f"  Columns (by '@id'):")
        for c in columns:
            print(f"    - {c if isinstance(c, str) else c.get('@id', c)}")
    print()

## 3. Data Extraction
Let's load each record set into a pandas DataFrame for analysis. We reference record sets by their `@id` field.

**Example**: To load the main protein abundance table, use its `@id`, e.g. `'cr:recordSet_1'` if that is the correct one obtained from the previous cell.

In [ ]:
# Gather all record set '@id's
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

# For demonstration, pick the largest DataFrame (most records):
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nUsing record set for analysis: {main_record_set_id}")
    print("Sample data:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter by a numeric field, normalize it, and group by a categorical field. Field names are referenced by their `@id` (column keys in the DataFrame).

In [ ]:
# Display available columns
main_df = dataframes[main_record_set_id]
print("Available columns (fields):")
print(main_df.columns.tolist())

# Pick example numeric and group field '@id'. Replace with true values based on your data.
# For demonstration, we try common mass spec names; adjust if needed.
import re
# Try to infer numeric fields (those containing "abundance", "count", "mw", "molecular_weight", "coverage" etc.)
numeric_field_candidates = [col for col in main_df.columns if re.search(r'(abundance|count|cover|mw|molecular_weight|peptide|amount|score)', col, re.IGNORECASE)]
if not numeric_field_candidates:
    numeric_field = main_df.columns[0]  # fallback
else:
    numeric_field = numeric_field_candidates[0]
print(f"Using numeric field for analysis: {numeric_field}")
group_field_candidates = [col for col in main_df.columns if re.search(r'(description|name|group|type|category|protein)', col, re.IGNORECASE)]
group_field = group_field_candidates[0] if group_field_candidates else None
print(f"Using group (categorical) field: {group_field}")

# Clean numeric column (convert to float, coerce errors)
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

# Filter to values above threshold (10)
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} ({len(filtered_df)} records):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() or 1)

print(f"Normalized '{numeric_field}' sample:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field if present
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    grouped_df[f'{numeric_field}_count'] = filtered_df.groupby(group_field)[numeric_field].count()
    print(f"Grouped mean and count by '{group_field}':")
    display(grouped_df.head())
else:
    print("No suitable categorical group field for group-by analysis.")

## 5. Visualization
Visualize a distribution of the selected numeric field and, if relevant, group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot distribution of the main numeric field
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30, color='royalblue')
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Optional: Boxplot by group if group_field exists
if group_field and group_field in filtered_df.columns:
    # Top N categories only for readability
    top_groups = filtered_df[group_field].value_counts().index[:8]
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=filtered_df[filtered_df[group_field].isin(top_groups)],
        x=group_field,
        y=numeric_field,
        palette="pastel"
    )
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No group field available for boxplot.")

## 6. Conclusion
In this notebook, we demonstrated how to access and perform basic exploration and analysis on a Croissant-formatted dataset using the `mlcroissant` library:
- Loaded metadata and records from the publicly available mass spectrometry dataset
- Explored record sets and the structure using `@id` referencing for fields
- Performed basic filtering, normalization, and aggregations (e.g., group-bys)
- Visualized key distributions of measured attributes

You can now proceed with domain-specific analyses, machine learning modeling, or further protein data interpretation!

**References**
- [FAIR² dataset page](https://doi.org/10.71728/senscience.vq4a-28xa)
- [MLCommons Croissant](https://mlcommons.github.io/croissant/)
